<a href="https://colab.research.google.com/github/SamyElnagar11/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamyElnagar11/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule idea:** rank visible pages (500+ impressions, average position above 0 and up to 20) whose CTR sits below the typical CTR of pages at a similar position, and send them for a title/meta review.

**Signal 1 (CTR vs position, linked to FlyRank's CTR-fix logic): MIXED.** In this dataset, median CTR falls from 0.23 (positions 3-10, n=8,660) to 0.15 (10-20, n=5,876), 0.06 (20-50, n=6,037) and 0.00 (50+, n=878). But positions 0-3 (n=555) have a lower median, 0.19, than 3-10, so the pattern is not clean at the very top. I compare each page only with pages in its own position bucket.

**Signal 2 (impressions vs CTR): CONFIRMED.** In this dataset, 87.4% of pages with up to 100 impressions (n=6,801) have a CTR of zero, against 33.0% for 500-2,000 impressions (n=6,502) and 0.6% above 10,000 (n=3,602). CTR is unreliable at low volume, so I only rank pages with at least 500 impressions.

Both are observed, directional patterns from one anonymized dataset, not claims about how Google works.

In [18]:
import pandas as pd, numpy as np
url = "https://raw.githubusercontent.com/SamyElnagar11/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)


v = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
v["pos_bucket"] = pd.cut(v["avg_position"], [0, 3, 10, 20, 50, np.inf], include_lowest=True)
print("SIGNAL 1: median CTR by position bucket")
print(v.groupby("pos_bucket", observed=True)["ctr"].agg(n="count", median_ctr="median").round(3))


d = df[df["avg_position"] > 0].copy()
d["imp_bucket"] = pd.cut(d["impressions_90d"], [0, 100, 500, 2000, 10000, np.inf])
print("\nSIGNAL 2: CTR by impressions bucket")
print(d.groupby("imp_bucket", observed=True)["ctr"].agg(
    n="count", median_ctr="median", share_zero_ctr=lambda s: (s == 0).mean()).round(3))


SIGNAL 1: median CTR by position bucket
                  n  median_ctr
pos_bucket                     
(-0.001, 3.0]   555        0.19
(3.0, 10.0]    8660        0.23
(10.0, 20.0]   5876        0.15
(20.0, 50.0]   6037        0.06
(50.0, inf]     878        0.00

SIGNAL 2: CTR by impressions bucket
                      n  median_ctr  share_zero_ctr
imp_bucket                                         
(0.0, 100.0]       6801        0.00           0.874
(100.0, 500.0]     5279        0.00           0.670
(500.0, 2000.0]    6502        0.12           0.330
(2000.0, 10000.0]  6611        0.19           0.055
(10000.0, inf]     3602        0.23           0.006


**Rule in plain words:** a page is eligible if it has at least 500 impressions and an average position above 0 and up to 20. Its score is its impressions times how far its CTR sits below the median CTR of pages in the same position bucket (0-3, 3-10, 10-20). Bigger score = more visibility with a bigger click gap.

**Reason code:** `low_ctr_for_position`. **Action label:** `review_title_meta`.

Inputs are current-window observed signals only. No product flags and no label-derived or future-window columns (such as `trend_direction`) are used.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import os

rule = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
rule["pos_bucket"] = pd.cut(rule["avg_position"], [0, 3, 10, 20], include_lowest=True)
rule["tier_median_ctr"] = rule.groupby("pos_bucket", observed=True)["ctr"].transform("median")
rule["ctr_gap"] = rule["tier_median_ctr"] - rule["ctr"]
rule["score"] = rule["impressions_90d"] * rule["ctr_gap"].clip(lower=0)

queue = rule[rule["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
queue["reason_code"] = "low_ctr_for_position"
queue["action"] = "review_title_meta"

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "content_id", "impressions_90d", "avg_position", "ctr",
            "tier_median_ctr", "score", "reason_code", "action"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("eligible pages:", len(rule), "| pages in queue:", len(queue))
queue[[c for c in out_cols if c != "content_id"]].head(10).round(3)

eligible pages: 12023 | pages in queue: 5896


,rank,impressions_90d,avg_position,ctr,tier_median_ctr,score,reason_code,action
0,1,295097,7.3,0.05,0.24,56068.43,low_ctr_for_position,review_title_meta
1,2,517715,4.2,0.14,0.24,51771.50,low_ctr_for_position,review_title_meta
2,3,208678,9.7,0.00,0.24,50082.72,low_ctr_for_position,review_title_meta
3,4,223271,7.8,0.03,0.24,46886.91,low_ctr_for_position,review_title_meta
4,5,272144,2.3,0.03,0.20,46264.48,low_ctr_for_position,review_title_meta
5,6,140079,7.6,0.01,0.24,32218.17,low_ctr_for_position,review_title_meta
6,7,213963,4.7,0.10,0.24,29954.82,low_ctr_for_position,review_title_meta
7,8,159590,7.8,0.06,0.24,28726.20,low_ctr_for_position,review_title_meta
8,9,134055,7.5,0.03,0.24,28151.55,low_ctr_for_position,review_title_meta
9,10,119217,7.0,0.02,0.24,26227.74,low_ctr_for_position,review_title_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 review.** For each of the top 20 pages I list the action, the reason code, a confidence note, and what would make the call wrong. Confidence is "higher" when the page has at least 5,000 impressions (the CTR is more stable) and "lower" otherwise. The "wrong if" lines come from simple checks I wrote and are a starting point for a human reviewer, not a verdict on any page.

In [20]:
def wrong_if(r):
    if r["ctr"] == 0:
        return "a CTR of exactly 0 on this much volume may be a tracking or data issue, not a title problem"
    if r["avg_position"] <= 3:
        return "it already ranks top 3, so low CTR may come from the results-page layout, not the title"
    if r["impressions_90d"] < 1500:
        return "volume is modest, so the CTR gap could be noise"
    return "the cause is intent mismatch or branded searches, not the title or meta"

top20 = queue.head(20)
for _, r in top20.iterrows():
    conf = "higher" if r["impressions_90d"] >= 5000 else "lower"
    print(f"#{int(r['rank'])} | action: {r['action']} | reason: {r['reason_code']} | confidence: {conf} "
          f"| CTR {r['ctr']:.2f} vs bucket median {r['tier_median_ctr']:.2f} at position {r['avg_position']:.1f}, "
          f"{int(r['impressions_90d'])} impressions | wrong if: {wrong_if(r)}")


#1 | action: review_title_meta | reason: low_ctr_for_position | confidence: higher | CTR 0.05 vs bucket median 0.24 at position 7.3, 295097 impressions | wrong if: the cause is intent mismatch or branded searches, not the title or meta
#2 | action: review_title_meta | reason: low_ctr_for_position | confidence: higher | CTR 0.14 vs bucket median 0.24 at position 4.2, 517715 impressions | wrong if: the cause is intent mismatch or branded searches, not the title or meta
#3 | action: review_title_meta | reason: low_ctr_for_position | confidence: higher | CTR 0.00 vs bucket median 0.24 at position 9.7, 208678 impressions | wrong if: a CTR of exactly 0 on this much volume may be a tracking or data issue, not a title problem
#4 | action: review_title_meta | reason: low_ctr_for_position | confidence: higher | CTR 0.03 vs bucket median 0.24 at position 7.8, 223271 impressions | wrong if: the cause is intent mismatch or branded searches, not the title or meta
#5 | action: review_title_meta | rea

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks (from reading my top 20):**
- **#3** (CTR 0.00 on 208,678 impressions at position 9.7): a CTR of exactly zero on this much volume looks more like a tracking or data issue than a title problem, so I would check the data before any rewrite.
- **#5 and #17** (positions 2.3 and 2.2, CTR 0.03 and 0.01): they already rank near the top, so the low CTR may come from the results-page layout (ads, snippets, other features), not the title or meta.
- **#14** (CTR 0.15 vs a bucket median of 0.20, 509,252 impressions) and **#2** (CTR 0.14 vs 0.24): their gap is modest and they rank high mainly because of volume.

**What I noticed about the baseline itself:**
- The score is impressions times the CTR gap, so very large pages rank high even when the gap is small. A version that also looks at the size of the gap alone could change the order.
- The confidence note was "higher" for all 20 pages, because every top page has more than 5,000 impressions, so it does not separate them yet.
- 15 of the 20 "wrong if" lines are the same generic sentence, because the rule is simple. A human reviewer should check each page.
- The bucket medians come from the same pages I rank, so the rule only says "below its peers", not "has a bad title".

**Leakage check:** the score uses only `impressions_90d`, `avg_position` and `ctr` from the current window. I did not use `trend_direction`, any FlyRank product flag, or any future-window column.

**Lane:** I confirm Lane 4 (CTR / Engagement Opportunity Scoring) for Week 5.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.